In [1]:
%pip install scipy
# %pip install dataclasses
%pip install pandas
%pip install numpy
%pip install torch
%pip install wandb
%pip install datasets
%pip install transformers
%pip install huggingface_hub
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [3]:
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'eval/accuracy', 'goal': 'maximize'},
    'parameters': {
        'batch_size':        {'values': [16, 32]},
        'weight_decay':      {'min': 0.0,  'max': 0.1},
        'warmup_ratio':      {'min': 0.0,  'max': 0.2},
        'backbone_lr':       {'distribution': 'log_uniform_values', 'min': 1e-6, 'max': 5e-5},
        'head_lr':           {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 5e-3},
        'unfreeze_at_step':  {'values': [100, 200, 500]},
        'label_smoothing':   {'distribution': 'uniform', 'min': 0.0, 'max': 0.2},
        'hidden_dropout':    {'values': [0.0, 0.1, 0.2]},
    }
}

In [5]:
import os
from datetime import datetime
from typing import Any, Dict, List
from collections import Counter

import numpy as np
import torch
import wandb
import evaluate

from datasets import load_dataset, Audio, load_from_disk
from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor,
    AutoConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from huggingface_hub import login
from google.colab import userdata

# ──────────────────────────────────────────────
# 1. CONSTANTS (never swept)
# ──────────────────────────────────────────────
model_id         = "utter-project/mHuBERT-147"
max_duration     = 3          # reduced from 7 to avoid OOM
sampling_rate    = 16_000
num_train_epochs = 3
gradient_accumulation_steps = 2
early_stopping_patience = 3
input_features_key = "input_values"   # w2v-bert would use "input_features"

AVAILABLE_CPU = int(userdata.get("REQUEST_CPU") or __import__('multiprocessing').cpu_count())
WANDB_TEAM    = userdata.get("WANDB_TEAM")
WANDB_PROJECT = userdata.get("WANDB_PROJECT")

# ──────────────────────────────────────────────
# 2. ONE-TIME SETUP  (login, data, feature extractor)
# ──────────────────────────────────────────────
login(token=userdata.get('HF_TOKEN'))
wandb.login(key=userdata.get('WANDB_TOKEN'))

feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True, return_attention_mask=True
)

dataset  = load_dataset("badrex/nnti-dataset-full")
train_ds = dataset['train'].shuffle(seed=42).cast_column("audio_filepath", Audio(sampling_rate=sampling_rate))
valid_ds = dataset['validation'].shuffle(seed=42).cast_column("audio_filepath", Audio(sampling_rate=sampling_rate))

LABELS     = train_ds.unique('language')
str_to_int = {s: i for i, s in enumerate(LABELS)}
int_to_str = {i: s for s, i in str_to_int.items()}
num_labels = len(LABELS)

# ──────────────────────────────────────────────
# 3. PREPROCESSING  (cached per max_duration)
# ──────────────────────────────────────────────
def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio_filepath"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        truncation=True,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        return_attention_mask=True,
    )
    inputs["label"]  = [str_to_int[x] for x in examples["language"]]
    inputs[input_features_key] = [np.array(x) for x in inputs[input_features_key]]
    inputs["length"] = [len(f) for f in inputs[input_features_key]]
    return inputs

keep_cols = ['speaker_id', 'language']
cache_path = f"./processed_data/encoded_{model_id.split('/')[-1]}_{max_duration}sec"

if os.path.exists(cache_path):
    print(f"Loading cached data from {cache_path}")
    train_ds_encoded = load_from_disk(cache_path + '/train')
    valid_ds_encoded = load_from_disk(cache_path + '/valid')
else:
    print("Preprocessing data...")
    map_kwargs = dict(
        batched=True, batch_size=32, num_proc=AVAILABLE_CPU,
        remove_columns=[c for c in train_ds.column_names if c not in keep_cols],
    )
    train_ds_encoded = train_ds.map(preprocess_function, **map_kwargs)
    valid_ds_encoded = valid_ds.map(preprocess_function, **map_kwargs)
    train_ds_encoded.save_to_disk(cache_path + '/train')
    valid_ds_encoded.save_to_disk(cache_path + '/valid')

# ──────────────────────────────────────────────
# 4. HELPERS
# ──────────────────────────────────────────────
class AudioDataCollator:
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch = {
            input_features_key: [f[input_features_key] for f in features],
            "attention_mask":   [f["attention_mask"]   for f in features],
        }
        batch = feature_extractor.pad(batch, padding=True, return_tensors="pt")
        batch["labels"] = torch.tensor([f["label"] for f in features], dtype=torch.long)
        return batch

class UnfreezeCallback(TrainerCallback):
    def __init__(self, unfreeze_at_step: int):
        self.unfreeze_at_step = unfreeze_at_step

    def on_step_begin(self, args, state, control, model=None, **kwargs):
        if state.global_step == self.unfreeze_at_step:
            for param in model.parameters():
                param.requires_grad = True
            print(f"✓ Backbone unfrozen at step {state.global_step}")

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=eval_pred.label_ids)

# ──────────────────────────────────────────────
# 5. SWEEP TRAIN FUNCTION  ← called once per sweep run
# ──────────────────────────────────────────────
def train():
    # Each sweep agent calls wandb.init() automatically before train(),
    # so wandb.config is already populated here.
    run = wandb.init()         # no-op if already initialised by the agent
    cfg = wandb.config         # hyperparameters for this trial

    current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")

    # --- build model fresh for every trial ---
    model_config = AutoConfig.from_pretrained(model_id)
    model_config.num_labels  = num_labels
    model_config.label2id    = str_to_int
    model_config.id2label    = int_to_str
    model_config.hidden_dropout    = cfg.hidden_dropout
    model_config.attention_dropout = cfg.hidden_dropout
    model_config.activation_dropout = cfg.hidden_dropout
    model_config.feat_proj_dropout = cfg.hidden_dropout

    model = AutoModelForAudioClassification.from_pretrained(model_id, config=model_config)

    # freeze backbone initially
    for param in model.hubert.parameters():
        param.requires_grad = False

    # --- optimizer with differential LRs ---
    backbone_params = [p for n, p in model.named_parameters()
                       if "classifier" not in n and "projector" not in n]
    head_params     = [p for n, p in model.named_parameters()
                       if "classifier" in n or "projector" in n]

    steps_per_epoch   = len(train_ds_encoded) // (cfg.batch_size * gradient_accumulation_steps)
    num_total_steps   = steps_per_epoch * num_train_epochs
    num_warmup_steps  = int(cfg.warmup_ratio * num_total_steps)

    optimizer = AdamW([
        {"params": backbone_params, "lr": cfg.backbone_lr},
        {"params": head_params,     "lr": cfg.head_lr},
    ], weight_decay=cfg.weight_decay)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_total_steps,
    )

    # --- training args ---
    training_args = TrainingArguments(
        output_dir=f"./sweep_runs/{run.name}",
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        weight_decay=cfg.weight_decay,
        label_smoothing_factor=cfg.label_smoothing,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        logging_steps=1,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        save_total_limit=1,
        fp16=True,
        group_by_length=True,
        report_to="wandb",
        dataloader_num_workers=AVAILABLE_CPU,
        dataloader_pin_memory=True,
        push_to_hub=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds_encoded,
        eval_dataset=valid_ds_encoded,
        processing_class=feature_extractor,
        data_collator=AudioDataCollator(),
        compute_metrics=compute_metrics,
        optimizers=(optimizer, scheduler),
        callbacks=[
            UnfreezeCallback(unfreeze_at_step=cfg.unfreeze_at_step),
            EarlyStoppingCallback(early_stopping_patience=early_stopping_patience),
        ],
    )

    trainer.train()
    wandb.finish()

# ──────────────────────────────────────────────
# 6. LAUNCH SWEEP
# ──────────────────────────────────────────────
# sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_TEAM)
existing_sweep_id = "mxs88lqp"
full_sweep_path = f"{WANDB_TEAM}/{WANDB_PROJECT}/{existing_sweep_id}"
print(f"Sweep ID: {existing_sweep_id}")

# count=N controls how many trials this agent runs
wandb.agent(full_sweep_path, function=train, count=7)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Loading cached data from ./processed_data/encoded_mHuBERT-147_3sec
Sweep ID: mxs88lqp


wandb: Agent Starting Run: rub0otcc with config:
wandb: 	backbone_lr: 1.596053054014257e-06
wandb: 	batch_size: 32
wandb: 	head_lr: 0.00039609906895993186
wandb: 	hidden_dropout: 0.1
wandb: 	label_smoothing: 0.0469697344490317
wandb: 	unfreeze_at_step: 100
wandb: 	warmup_ratio: 0.008264009371119197
wandb: 	weight_decay: 0.01085998327864659
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

HubertForSequenceClassification LOAD REPORT from: utter-project/mHuBERT-147
Key               | Status  | 
------------------+---------+-
projector.weight  | MISSING | 
projector.bias    | MISSING | 
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
wandb: WARNING Config item 'hidden_dropout' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'warmup_ratio' was locked by 'sweep' (ignored update).


Step,Training Loss,Validation Loss,Accuracy
100,5.997423,3.057644,0.078485
200,5.879001,3.049573,0.085455
300,5.859941,3.049156,0.084545
400,5.865068,3.049006,0.084848


... (78 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

eval/accuracy,▁█▇▇
eval/loss,█▅▁▂
eval/runtime,▆▂█▁
eval/samples_per_second,▃▇▁█
eval/steps_per_second,▃▇▁█
train/epoch,▁▁▁▁▁▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
train/grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▁ ▂▃▁▂ ▃▂▃▃█▂▄▇
train/learning_rate,▂▃▄▅▅████▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁
train/loss,███████▇▇▇▇▇▇▇▆▆▆▆▄▅▄▃▄▂▃▁▁▄▄▂▄▄▃▃▄▅▄▂▃▄
eval/accuracy,0.13667


In [ ]:
from google.colab import drive
import shutil
import os

try:
  # 1. Монтируем диск
  drive.mount('/content/drive')

  # 2. Создаем папку на Диске
  save_path_drive = "/content/drive/MyDrive/my_sweep_results"
  os.makedirs(save_path_drive, exist_ok=True)

  # 3. Сохраняем модель (если вы это делаете ВНЕ функции train, после завершения sweep)
  # Допустим, ваша лучшая модель сохранилась локально в "./indic-SLID/inprogress"
  if os.path.exists("./sweep_runs"):
      shutil.copytree("./sweep_runs", f"{save_path_drive}/best_model_final", dirs_exist_ok=True)
      print("Веса успешно скопированы на Google Drive!")
except Exception as e:
    print(f"Произошла ошибка во время Sweep: {e}")
finally:
    print("Завершаю работу рантайма...")
    from google.colab import runtime
    runtime.unassign()